# 03 — Modo directo: Control del potenciostato

El **modo directo** te permite controlar el potenciostato manualmente — estableciendo potencial o corriente y leyendo mediciones al instante — sin cargar un archivo de método.

Usa el modo directo para:
- Mediciones puntuales rápidas (OCV, mantener potencial)
- Pasos de acondicionamiento o equilibrado
- Solución de problemas de un dispositivo o configuración de celda
- Construir bucles de medición personalizados en Python

Para secuencias de experimentos automatizados (LSV, CV, EIS…) ver **`06_method_mode.ipynb`**.

### Requisitos previos
- IviumSoft en ejecución
- Dispositivo conectado (ver `02_device_and_instance_management.ipynb`)
- Referencia de manejo de errores: `01_getting_started.ipynb` — Sección 4

In [ ]:
import time
import matplotlib.pyplot as plt
from pyvium import Pyvium

Pyvium.open_driver()
Pyvium.connect_device()
print("Ready")

## 1. Modos de conexión

El modo de conexión determina el cableado de los electrodos. Configúralo siempre antes de encender la celda.

| Código | Modo | Caso de uso |
|------|------|----------|
| 0 | Off | Sin conexión |
| 1 | EStat 4EL | Potenciostato estándar de 4 electrodos (por defecto) |
| 2 | EStat 2EL | Potenciostato de 2 electrodos |
| 3–6 | EStat Dummy | Celda ficticia (pruebas/calibración) |
| 7 | IStat 4EL | Galvanostato de 4 electrodos |
| 8 | IStat 2EL | Galvanostato de 2 electrodos |
| 9 | IStat Dummy | Celda ficticia galvanostato |
| 10 | BiStat 4EL | Bipotenciostato 4 electrodos |
| 11 | BiStat 2EL | Bipotenciostato 2 electrodos |

In [ ]:
# Configurar modo potenciostato estándar de 4 electrodos
Pyvium.set_connection_mode(1)
print("Connection mode: EStat 4EL (standard potentiostat)")

## 2. Estado de la celda

`get_cell_status()` devuelve una lista de indicadores de estado activos decodificados de un bitmask. Los indicadores y sus bits correspondientes:

| Indicador | Bit | Significado |
|------|-----|-------|
| `'Cell on'` | — | La salida de la celda está activa |
| `'Cell off'` | — | La salida de la celda está desactivada (por defecto tras conectar) |
| `'I_ovl'` | 2 | Sobrecarga de corriente — reducir el rango de corriente |
| `'Anin1_ovl'` | 4 | Sobrecarga de entrada analógica 1 |
| `'E_ovl'` | 5 | Sobrecarga de potencial — reducir el rango de potencial |
| `'CellOff_button pressed'` | 7 | Se presionó el botón de seguridad hardware |

Cuando `'I_ovl'` o `'E_ovl'` están activos la medición no es confiable — verifica tu rango de corriente y el potencial aplicado.

> **Importante:** Las configuraciones de modo directo (potencial, rango de corriente, filtro, estabilidad) se **suspenden** mientras un método está en ejecución vía `start_method()`. Se reanudan cuando el método termina o se cancela. Configura los ajustes de modo directo solo cuando no haya ningún método activo.

In [ ]:
status = Pyvium.get_cell_status()
print(f"Cell status flags: {status}")

# Verificar sobrecargas antes de confiar en las mediciones
OVERLOAD_FLAGS = {'I_ovl', 'E_ovl', 'Anin1_ovl'}
overloads = OVERLOAD_FLAGS & set(status)
if overloads:
    print(f"WARNING: overload detected: {overloads}")
else:
    print("No overloads — measurements are reliable")

## 3. Rangos de corriente

Establece el rango de corriente **antes** de encender la celda. Un rango demasiado grande da poca resolución; uno demasiado pequeño causará una sobrecarga.

La DLL define los códigos de rango de corriente como una secuencia decreciente de décadas. Según la referencia de la DLL de IviumSoft, `IV_setcurrentrange` comienza en **0 = 10 A, 1 = 1 A**, y continúa hacia abajo en décadas:

| Código | Rango (IviumStat / dispositivos de rango completo) |
|------|----------------------------------------------|
| 0 | 10 A |
| 1 | 1 A |
| 2 | 100 mA |
| 3 | 10 mA |
| 4 | 1 mA |
| 5 | 100 µA |
| 6 | 10 µA |
| 7 | 1 µA |
| 8 | 100 nA |
| … | … |

Los dispositivos de baja corriente (CompactStat, pocketSTAT) solo admiten un subconjunto de los rangos inferiores — los códigos siguen el mismo patrón decreciente de décadas pero el rango máximo del dispositivo es inferior a 10 A, por lo que el código 0 mapea al rango disponible más alto para ese modelo.

> **Siempre verifica el código correcto para tu dispositivo** mirando el menú desplegable de rango de corriente en modo directo de IviumSoft antes de usar estos valores en un script.

> Para WE2 (BiStat), la escala es diferente: `IV_setcurrentrangeWE2` comienza en 0 = 10 mA, 1 = 1 mA, etc. Ver notebook `05_bipotentiostat_and_we32.ipynb`.

In [ ]:
Pyvium.set_current_range(4)  # rango de 1 mA
print("Current range set to 1 mA")

## 4. Configuración de filtro y estabilidad

El **filtro** controla el ancho de banda de la medición de corriente. Menor ancho de banda = menos ruido pero respuesta más lenta.

| Código | Filtro |
|------|------|
| 0 | 1 MHz (más rápido, más ruido) |
| 1 | 100 kHz |
| 2 | 10 kHz |
| 3 | 1 kHz |
| 4 | 10 Hz (más lento, menos ruido) |

La **estabilidad** ajusta el bucle de retroalimentación del potenciostato para la impedancia de la celda:

| Código | Estabilidad | Usar cuando |
|------|-----------|----------|
| 0 | Alta velocidad | Celdas de baja impedancia, barridos rápidos |
| 1 | Estándar | La mayoría de las celdas electroquímicas (por defecto) |
| 2 | Alta estabilidad | Celdas de alta impedancia, señales ruidosas |

In [ ]:
Pyvium.set_filter(2)       # filtro de 10 kHz
Pyvium.set_stability(1)    # estabilidad estándar
print("Filter: 10 kHz | Stability: Standard")

## 5. Encender la celda y establecer el potencial

> **Nota de seguridad:** Siempre establece el potencial deseado **antes** de encender la celda. Esto evita un pico transitorio a un valor inesperado.

In [ ]:
# Establecer potencial ANTES de encender la celda
Pyvium.set_potential(0.1)  # 0.1 V vs referencia
print("Target potential: 0.1 V")

Pyvium.set_cell_on()
status = Pyvium.get_cell_status()
print(f"Cell status: {status}")
print(f"Potential: {Pyvium.get_potential()}")

## 6. Leer potencial y corriente

Después de encender la celda, lee potencial y corriente. Dale al potenciostato un breve tiempo de asentamiento después de cambiar el potencial.

In [ ]:
time.sleep(0.1)  # permitir asentamiento

potential = Pyvium.get_potential()
current   = Pyvium.get_current()

print(f"Measured potential : {potential:.6f} V")
print(f"Measured current   : {current:.3e} A")

## 7. Mantenimiento potenciostático — Muestreo en el tiempo

Un patrón común: mantener el potencial fijo y registrar corriente vs. tiempo (cronoamperometría).

In [ ]:
HOLD_POTENTIAL = 0.2   # V
DURATION_S     = 5.0   # segundos
INTERVAL_S     = 0.2   # intervalo de muestreo

Pyvium.set_potential(HOLD_POTENTIAL)
Pyvium.set_cell_on()
time.sleep(0.1)  # asentamiento

timestamps = []
currents   = []
t_start    = time.time()

while (time.time() - t_start) < DURATION_S:
    t = time.time() - t_start
    I = Pyvium.get_current()
    timestamps.append(t)
    currents.append(I * 1e6)  # convertir a µA
    time.sleep(INTERVAL_S)

Pyvium.set_cell_off()
print(f"Collected {len(timestamps)} points over {DURATION_S:.0f} s")

# Graficar
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(timestamps, currents, 'b-o', markersize=3)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Current (µA)")
ax.set_title(f"Chronoamperometry — Hold at {HOLD_POTENTIAL} V")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Modo galvanostático (establecer corriente)

En modo galvanostático el instrumento controla la corriente y tú lees el potencial. Cambia primero a un modo de conexión IStat.

In [ ]:
HOLD_CURRENT = 1e-6  # 1 µA

Pyvium.set_connection_mode(7)   # IStat 4EL (galvanostato)
Pyvium.set_current_range(4)     # rango de 1 µA
Pyvium.set_current(HOLD_CURRENT)
Pyvium.set_cell_on()
time.sleep(0.1)

potential = Pyvium.get_potential()
current   = Pyvium.get_current()
print(f"Applied current  : {current:.3e} A")
print(f"Measured potential: {potential:.6f} V")

Pyvium.set_cell_off()
Pyvium.set_connection_mode(1)   # de vuelta al potenciostato

## 9. Potencial de circuito abierto (OCV)

El OCV se mide con la celda **apagada** — el instrumento solo escucha, no aplica ninguna señal.

In [ ]:
OCV_DURATION_S = 10.0
INTERVAL_S     = 0.5

Pyvium.set_cell_off()  # la celda debe estar apagada para OCV verdadero

timestamps = []
potentials = []
t_start    = time.time()

while (time.time() - t_start) < OCV_DURATION_S:
    t = time.time() - t_start
    E = Pyvium.get_potential()
    timestamps.append(t)
    potentials.append(E)
    time.sleep(INTERVAL_S)

print(f"OCV at end: {potentials[-1]:.4f} V")

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(timestamps, potentials, 'g-')
ax.set_xlabel("Time (s)")
ax.set_ylabel("OCV (V)")
ax.set_title("Open Circuit Potential vs. Time")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Limpieza

In [ ]:
Pyvium.set_cell_off()
Pyvium.disconnect_device()
Pyvium.close_driver()
print("Done")

---

## Resumen

| Tarea | Método |
|------|-------|
| Configurar cableado de electrodos | `set_connection_mode(n)` |
| Verificar estado de la celda | `get_cell_status()` |
| Establecer rango de corriente | `set_current_range(n)` |
| Configurar filtro de medición | `set_filter(n)` |
| Configurar estabilidad | `set_stability(n)` |
| Aplicar potencial (potenciostato) | `set_potential(V)` |
| Aplicar corriente (galvanostato) | `set_current(A)` |
| Leer potencial | `get_potential()` → float |
| Leer corriente | `get_current()` → float |
| Encender / apagar celda | `set_cell_on()` / `set_cell_off()` |

## Siguiente

- **`04_direct_mode_signals.ipynb`** — Trazas de alta velocidad, DAC/ADC, E/S digital, control AC
- **`05_bipotentiostat_and_we32.ipynb`** — BiStat (WE2) y arreglo multicanal WE32
- **`06_method_mode.ipynb`** — Ejecutar métodos electroquímicos completos (LSV, CV, EIS…)